# Phase 3 — Per-Venue Microstructure

Load the snapshot CSV produced by `scripts/fetch_snapshot.py` and visualize per-venue spread, depth, and book shape across the 3 NBA Finals pairs.

To refresh the snapshot:
```bash
uv run python scripts/fetch_snapshot.py
```

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.family"] = "monospace"

df = pd.read_csv("../data/processed/microstructure_snapshot.csv")
print(f"Loaded {len(df)} rows")
df

## Summary table — YES side, all 3 markets, both venues

In [ ]:
yes_df = df[df["side"] == "yes"].copy()
cols = ["venue", "market_id", "best_bid", "best_ask", "spread_abs", "spread_bps",
        "depth_top_of_book", "depth_within_1c", "n_bid_levels", "n_ask_levels"]
yes_df[cols].sort_values(["market_id", "venue"])

## Figure 1 — Spread comparison (bps) by market and venue

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
pivot = yes_df.pivot_table(index="market_id", columns="venue", values="spread_bps")
pivot.plot(kind="bar", ax=ax, color=["#2ecc71", "#9b59b6"])
ax.set_ylabel("Spread (bps of mid)")
ax.set_title("YES-side spread by venue, 3 NBA Finals markets")
ax.set_xlabel("")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig("../data/processed/fig_spread_by_venue.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 2 — Depth within 1¢ of mid

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
pivot = yes_df.pivot_table(index="market_id", columns="venue", values="depth_within_1c")
pivot.plot(kind="bar", ax=ax, color=["#2ecc71", "#9b59b6"])
ax.set_ylabel("Depth within ±1¢ of mid (contracts)")
ax.set_title("YES-side depth by venue, 3 NBA Finals markets")
ax.set_xlabel("")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig("../data/processed/fig_depth_1c.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 3 — Book shape (price levels populated)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
yes_df["total_levels"] = yes_df["n_bid_levels"] + yes_df["n_ask_levels"]
pivot = yes_df.pivot_table(index="market_id", columns="venue", values="total_levels")
pivot.plot(kind="bar", ax=ax, color=["#2ecc71", "#9b59b6"])
ax.set_ylabel("Total price levels (bids + asks)")
ax.set_title("Book depth (levels populated) by venue")
ax.set_xlabel("")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig("../data/processed/fig_book_levels.png", dpi=150, bbox_inches="tight")
plt.show()

## Side-by-side reading

The three figures together describe the per-venue microstructure at this snapshot. Phase 4 builds on this to compute cross-venue mid-price discrepancy and executable arb after fees.